<a href="https://colab.research.google.com/github/Carinaaa/ML-Learning-Path/blob/intro-LLM/Company_brochure_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import json
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import userdata

In [23]:
from bs4 import BeautifulSoup
import requests

headers = {'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'}

def fetch_website_contents(url):
  """
  Return the title and contents of the website at the given url;
  truncate to 2,000 characters as a sensible limit
  """
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.content, "html.parser")
  title = soup.title.string if soup.title else "No title found"
  if soup.body:
    for irrelevant in soup.body(["script","style","img","input"]):
      irrelevant.decompose()
    text = soup.body.get_text(separator='\n', strip=True)
  else:
    text = ""
  return (title + "\n\n" + text)[:2_000]

def fetch_website_links(url):
  """
  Return the links on the webiste at the given url
  I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
  Feel free to use a class and optimize it!
  """
  respons = requests.get(url, headers=headers)
  soup = BeautifulSoup(respons.content, "html.parser")
  links = [link.get("href") for link in soup.find_all("a")]
  return links



In [6]:
openai = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

In [13]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be the most relevant to include in a brouchure about the company, such as links to an About page, or a Company page, or Careers/Job pages.
You should respond in JSON as in this example:

{
  "links": [
    {'type' : 'about page', 'url' : 'https://edwarddonner.com/about'},
    {'type' : 'company page', 'url' : 'https://edwarddonner.com/company'}
  ]
}
"""

In [9]:
def get_links_user_prompt(url):
  user_prompt = f"""
  Here is the list of the links on the website {url} -
  Please decide which of these are relevant web link for a brochure about the company, respond with the full https URL in JSON format.
  Do not include Terms of Service, Privacy, email links.

  Links (some might be relative links):

  """
  links = fetch_website_links(url)
  user_prompt += "\n".join(links)
  return user_prompt

In [10]:
print(get_links_user_prompt("https://edwarddonner.com"))


  Here is the list of the links on the website https://edwarddonner.com -
  Please decide which of these are relevant web link for a brochure about the company, respond with the full https URL in JSON format.
  Do not include Terms of Service, Privacy, email links.

  Links (some might be relative links):
  
  https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws

In [16]:
MODEL = 'gpt-5-nano'

def select_relevant_links(url):
  response = openai.chat.completions.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": link_system_prompt},
          {"role": "user", "content": get_links_user_prompt(url)}
      ],
      response_format={"type": "json_object"},
  )
  result = response.choices[0].message.content
  links = json.loads(result)
  print(f"Found {len(links['links'])} relevant links")
  return links

In [17]:
select_relevant_links("https://edwarddonner.com")

Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [18]:
select_relevant_links("https://huggingface.co")

Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'zhihu', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'}]}

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [24]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
tencent/HunyuanVideo-1.5
Updated
2 days ago
•
243
•
487
facebook/sam3
Updated
2 days ago
•
50.4k
•
482
WeiboAI/VibeThinker-1.5B
Updated
8 days ago
•
14k
•
436
Supertone/supertonic
Updated
1 day ago
•
594
•
163
moonshotai/Kimi-K2-Thinking
Updated
15 days ago
•
218k
•
1.36k
Browse 1M+ models
Spaces
Running
on
Zero
MCP
Featured
1.19k
Qwen Image Edit Camera Control
🎬
Featured
1.19k
Fast 4 step inference with Qwen Image Edit 2509
Running
on
CPU Upgrade
Featured
2.37k
The Smol Training Playbook
📚
Featured
2.37k
The secrets to building world-class LLMs
Running
on
Zero
Featured
219
Depth Anything 3
🏢
Featured
219


In [25]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [26]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [27]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found 4 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\ntencent/HunyuanVideo-1.5\nUpdated\n2 days ago\n•\n243\n•\n487\nfacebook/sam3\nUpdated\n2 days ago\n•\n50.4k\n•\n482\nWeiboAI/VibeThinker-1.5B\nUpdated\n8 days ago\n•\n14k\n•\n436\nSupertone/supertonic\nUpdated\n1 day ago\n•\n594\n•\n163\nmoonshotai/Kimi-K2-Thinking\nUpdated\n15 days ago\n•\n218k\n•\n1.36k\nBrowse 1M+ models\nSpaces\nRunning\non\nZero\nMCP\nFeatured\n1.19k\nQwen Image Edit C

In [28]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [29]:
create_brochure("HuggingFace", "https://huggingface.co")

Found 7 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a leading AI company and community dedicated to building the future of machine learning. As a vibrant collaboration platform, it empowers machine learning engineers, scientists, and enthusiasts around the world to create, share, discover, and experiment with open-source models, datasets, and applications.

At the core of Hugging Face is a mission to **democratize good machine learning**, fostering an open, ethical, and collaborative environment that accelerates AI innovation globally.

---

## What Hugging Face Offers

- **Models:** Explore over 1 million pre-trained machine learning models supporting multiple modalities including text, image, video, audio, and 3D data.
- **Datasets:** Access a vast repository of 250,000+ datasets meticulously curated and updated regularly, enabling superior training and evaluation.
- **Spaces:** Host and collaborate on ML applications seamlessly via an interactive platform that supports public model sharing and app deployment.
- **Community:** Join a fast-growing community of over 190 team members and tens of thousands of contributors pushing the boundaries of AI.
- **Open Source Stack:** Utilize the Hugging Face open-source ecosystem to speed up development and accelerate your AI projects.
- **Enterprise Solutions:** Comprehensive enterprise-grade services and support for businesses looking to integrate and scale AI.

---

## Company Culture

Hugging Face fosters a culture grounded in **collaboration, openness, and continuous learning**. The company's community-first ethos encourages sharing knowledge and building an inclusive AI ecosystem. This culture is reflected in:

- Open-source commitment and transparent communication.
- Encouraging portfolio-building through public contributions.
- Emphasis on ethical AI and responsible innovation.
- Active engagement through forums, blogs, papers, and events.

---

## Customers & Impact

Hugging Face serves millions of users globally, including:

- Individual researchers and hobbyists building AI projects.
- Academic institutions advancing the science of machine learning.
- Industry leaders and enterprises integrating cutting-edge AI technologies.
- Developers deploying AI-powered applications across diverse sectors like autonomous vehicles, speech recognition, and multimedia.

Their tools and datasets are widely adopted for both research and production, making Hugging Face a pivotal hub in the AI revolution.

---

## Careers & Opportunities

If you are passionate about AI and want to be part of shaping the future of machine learning, Hugging Face invites you to join their team. They are growing rapidly and continuously look for talented individuals who:

- Believe in democratizing AI and open science.
- Thrive in a collaborative, fast-paced environment.
- Are eager to innovate in open-source ML libraries, tooling, and research.

With 191+ team members and counting, Hugging Face offers opportunities in engineering, research, product, design, and operations.

**Join the mission:**  
> *“Democratize good machine learning, one commit at a time.”*  

Explore current job openings and contribute to the AI community at [huggingface.co](https://huggingface.co).

---

## Connect & Learn More

- Explore the platform and browse models & datasets: [huggingface.co](https://huggingface.co)
- Follow updates and community discussions on GitHub, Twitter, LinkedIn, and Discord.
- Read papers and blog articles on AI advancements and the Hugging Face ecosystem.
- Access extensive documentation and learning tracks to deepen your ML expertise.

---

**Hugging Face** – Building the future of AI with an open, collaborative community.

---

### Brand Colors  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280

---

In [30]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [31]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found 9 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the premier collaboration platform for the global machine learning (ML) community. It serves as a vibrant ecosystem where researchers, engineers, and enthusiasts build the future of AI together by sharing and experimenting with open-source ML models, datasets, and applications.

The platform supports a wide variety of modalities including text, image, video, audio, and even 3D data, enabling innovation across multiple domains of artificial intelligence.

---

## What We Offer

- **Hugging Face Hub**: A central repository hosting over 1 million machine learning models and 250,000 datasets, updated continuously by a fast-growing, engaged community.
- **Spaces**: A collaborative environment where developers deploy and showcase AI applications, with thousands of featured apps ranging from image editing and video generation to depth mapping and LLM training guides.
- **Open Source Stack**: Our technology stack accelerates ML development by providing accessible and reliable tools.
- **Community Collaboration**: Users can create, discover, and collaborate on public projects, while building their personal ML portfolio and professional presence.

---

## Our Customers and Community

Hugging Face empowers a diverse user base including:

- AI researchers and scientists pushing the boundaries of machine learning
- Machine learning engineers building production-ready models
- Enterprises adopting cutting-edge AI solutions for real-world challenges
- Developers and hobbyists eager to explore and innovate in AI

Key enterprises and organizations rely on Hugging Face to accelerate their AI projects, leveraging our rich library of models and datasets.

---

## Company Culture

- **Open and Ethical AI**: We are committed to fostering an open-source environment that promotes transparency, collaboration, and ethical use of AI technology.
- **Community-Driven Innovation**: Our fast-growing, supportive community fuels continuous innovation and knowledge sharing.
- **Multimodal Exploration**: Encouraging creativity across different data modalities — from text and images to video and 3D applications.
- **Empowerment and Growth**: We focus on empowering AI practitioners of all skill levels to learn, share, and succeed together.

---

## Careers at Hugging Face

Join a passionate, mission-driven team that is shaping the future of AI. Careers at Hugging Face include roles in:

- Machine Learning Engineering
- Research and Development
- Software Development
- Community Management
- Customer Success and Enterprise Solutions

If you are excited about open-source AI and want to work in a collaborative, innovative space, consider applying to Hugging Face to help build an open and ethical AI future.

---

## Get Involved

- Explore AI models, datasets, and applications on our platform.
- Share your work with a global community.
- Collaborate on projects and push the boundaries of AI.
- Sign up today and start building your AI portfolio at [huggingface.co](https://huggingface.co).

---

**Hugging Face** — Where the Machine Learning Community Builds the Future Together.

*Brand Colors: #FFD21E (Yellow), #FF9D00 (Orange), #6B7280 (Gray)*  
*Official logo and brand assets available on the Hugging Face website.*

---